# Serial Killers Dataset - Exploration Notebook

This notebook provides examples for exploring and analyzing the Serial Killers Dataset.

**Dataset**: 494 serial killer profiles from Wikipedia
**Source**: https://en.wikipedia.org/wiki/List_of_serial_killers_by_number_of_victims

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 100)

print("Libraries loaded successfully")

In [ ]:
# Load the CSV dataset (recommended for analysis)
df = pd.read_csv('serial_killers_dataset.csv', keep_default_na=False)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nEmpty string counts:\n{(df == '').sum().sort_values(ascending=False)}")

In [ ]:
# Basic statistics
print("=== VICTIM COUNT STATISTICS ===")
# Convert to numeric, treating empty strings as NaN
for col in ['victims_proven', 'victims_possible', 'victims_confessed', 'victim_count_max', 'active_start_year', 'active_end_year', 'years_active']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
print(df[['victims_proven', 'victims_possible', 'victims_confessed']].describe())

print("\n=== VICTIM COUNT CATEGORIES ===")
print(df['victim_count_category'].value_counts())

print("\n=== GENDER DISTRIBUTION ===")
print(df['gender'].value_counts())

print("\n=== TOP 15 COUNTRIES ===")
print(df['country_primary'].value_counts().head(15))

In [ ]:
# Top 20 by proven victims
top20 = df.nlargest(20, 'victims_proven')[['name', 'victims_proven', 'victims_possible', 'country_primary', 'active_start_year', 'active_end_year', 'gender']]
print(top20.to_string(index=False))

In [ ]:
# Visualization: Victim count distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram of proven victims
df['victims_proven'].dropna().astype(int).hist(bins=30, ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('Distribution of Proven Victims')
axes[0,0].set_xlabel('Number of Proven Victims')
axes[0,0].set_ylabel('Count')

# Victim count categories
df['victim_count_category'].value_counts().plot(kind='bar', ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Victim Count Categories')
axes[0,1].set_xlabel('Category')
axes[0,1].set_ylabel('Number of Killers')
axes[0,1].tick_params(axis='x', rotation=45)

# Top 10 countries
df['country_primary'].value_counts().head(10).plot(kind='barh', ax=axes[1,0], color='coral')
axes[1,0].set_title('Top 10 Countries by Number of Serial Killers')
axes[1,0].set_xlabel('Count')
axes[1,0].invert_yaxis()

# Gender distribution
df['gender'].value_counts().plot(kind='pie', ax=axes[1,1], autopct='%1.1f%%', startangle=90)
axes[1,1].set_title('Gender Distribution')
axes[1,1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Timeline analysis
timeline_df = df.dropna(subset=['active_start_year', 'active_end_year']).copy()
timeline_df['active_start_year'] = timeline_df['active_start_year'].astype(int)
timeline_df['active_end_year'] = timeline_df['active_end_year'].astype(int)
timeline_df['years_active'] = timeline_df['active_end_year'] - timeline_df['active_start_year']

print(f"Killers with timeline data: {len(timeline_df)}")
print(f"Average years active: {timeline_df['years_active'].mean():.1f}")
print(f"Median years active: {timeline_df['years_active'].median():.1f}")

# Decade analysis
timeline_df['decade'] = (timeline_df['active_start_year'] // 10) * 10
decade_counts = timeline_df['decade'].value_counts().sort_index()
print("\nKillers by starting decade:")
print(decade_counts)

In [ ]:
# Visualization: Timeline trends
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Killers by decade
decade_counts.plot(kind='bar', ax=axes[0], color='mediumseagreen')
axes[0].set_title('Serial Killers by Starting Decade')
axes[0].set_xlabel('Decade')
axes[0].set_ylabel('Number of Killers')
axes[0].tick_params(axis='x', rotation=45)

# Years active distribution
timeline_df['years_active'].hist(bins=20, ax=axes[1], edgecolor='black', color='gold')
axes[1].set_title('Distribution of Years Active')
axes[1].set_xlabel('Years Active')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Special categories analysis
print("=== MEDICAL PROFESSIONALS ===")
medical = df[df['medical_professional'] == True]
print(f"Count: {len(medical)}")
if len(medical) > 0:
    print(medical[['name', 'victims_proven', 'country_primary', 'occupation']].to_string(index=False))

print("\n=== LAW ENFORCEMENT ===")
le = df[df['law_enforcement'] == True]
print(f"Count: {len(le)}")
if len(le) > 0:
    print(le[['name', 'victims_proven', 'country_primary', 'occupation']].to_string(index=False))

print("\n=== GROUPS/COUPLES ===")
groups = df[df['operated_alone'] == False]
print(f"Count: {len(groups)}")
if len(groups) > 0:
    print(groups[['name', 'victims_proven', 'country_primary']].to_string(index=False))

In [ ]:
# Female serial killers analysis
female = df[df['gender'] == 'Female']
print(f"Female serial killers: {len(female)}")
if len(female) > 0:
    print(female.nlargest(10, 'victims_proven')[['name', 'victims_proven', 'country_primary', 'active_start_year', 'active_end_year']].to_string(index=False))

# Compare male vs female victim counts
male = df[df['gender'] == 'Male']
print(f"\nMale avg proven victims: {male['victims_proven'].mean():.1f}")
print(f"Female avg proven victims: {female['victims_proven'].mean():.1f}")
print(f"Male median proven victims: {male['victims_proven'].median():.1f}")
print(f"Female median proven victims: {female['victims_proven'].median():.1f}")

In [ ]:
# Correlation analysis (numeric fields only)
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix of Numeric Fields')
plt.tight_layout()
plt.show()

In [ ]:
# Load full JSON for detailed analysis
with open('serial_killers_dataset.json', 'r', encoding='utf-8') as f:
    full_data = json.load(f)

print(f"Full JSON records: {len(full_data)}")

# Example: Access detailed sections for a specific killer
garavito = next((k for k in full_data if k['name'] == 'Luis Garavito'), None)
if garavito:
    print(f"\n=== Luis Garavito - Full Details ===")
    print(f"Summary: {garavito['summary'][:300]}...")
    print(f"\nInfobox keys: {list(garavito['infobox'].keys())}")
    print(f"\nSections: {list(garavito['sections'].keys())}")
    print(f"\nModus Operandi: {garavito['modus_operandi'][:500]}...")
    print(f"\nCategories: {garavito['categories']}")

In [ ]:
# Advanced: Text analysis of modus operandi
from collections import Counter
import re

# Extract common words from modus operandi descriptions
mo_texts = df['modus_operandi'].replace('', np.nan).dropna()
all_words = []
for text in mo_texts:
    words = re.findall(r'\b[a-z]{4,}\b', text.lower())
    all_words.extend(words)

# Filter common stop words
stop_words = {'the', 'and', 'was', 'were', 'with', 'from', 'that', 'this', 'have', 'has', 'had', 'been', 'into', 'they', 'their', 'there', 'then', 'than', 'when', 'where', 'while', 'after', 'before', 'during', 'under', 'over', 'through', 'between', 'among', 'against', 'within', 'without', 'about', 'above', 'below', 'upon', 'since', 'until', 'unless', 'because', 'although', 'however', 'therefore', 'moreover', 'furthermore', 'nevertheless', 'otherwise', 'instead', 'meanwhile', 'according', 'accordingly', 'actually', 'apparently', 'basically', 'certainly', 'clearly', 'definitely', 'eventually', 'finally', 'generally', 'hopefully', 'ideally', 'ironically', 'literally', 'naturally', 'obviously', 'presumably', 'probably', 'really', 'simply', 'supposedly', 'surprisingly', 'typically', 'ultimately', 'usually', 'virtually'}
filtered_words = [w for w in all_words if w not in stop_words]
word_counts = Counter(filtered_words).most_common(30)

print("=== Most Common Words in Modus Operandi ===")
for word, count in word_counts:
    print(f"  {word}: {count}")

# Visualize
words, counts = zip(*word_counts[:20])
plt.figure(figsize=(10, 6))
plt.barh(range(len(words)), counts, color='teal')
plt.yticks(range(len(words)), words)
plt.xlabel('Frequency')
plt.title('Top 20 Words in Modus Operandi Descriptions')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Machine Learning Example: Predict victim count category
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Prepare features - use only rows with complete data
ml_df = df.dropna(subset=['victims_proven', 'country_primary', 'gender', 'active_start_year']).copy()
ml_df['victim_category'] = pd.cut(ml_df['victims_proven'], 
                                   bins=[0, 5, 15, 30, 300], 
                                   labels=['Very Low', 'Low', 'Medium', 'High'])

# Encode categorical variables
le_country = LabelEncoder()
le_gender = LabelEncoder()
ml_df['country_encoded'] = le_country.fit_transform(ml_df['country_primary'])
ml_df['gender_encoded'] = le_gender.fit_transform(ml_df['gender'])

# Features and target
X = ml_df[['country_encoded', 'gender_encoded', 'active_start_year', 'operated_alone', 'medical_professional', 'law_enforcement']]
y = ml_df['victim_category']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

# Feature importance
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print("\n=== Feature Importance ===")
print(importance_df.to_string(index=False))

## 🎯 Next Steps

1. **Deep dive** into specific killers using the full JSON data
2. **Geographic analysis** - map killer locations
3. **Temporal trends** - analyze patterns over decades
4. **Behavioral clustering** - group by MO similarities
5. **Victimology studies** - analyze victim demographics
6. **Comparative studies** - solo vs group, male vs female, etc.

## 📚 Resources

- [DATASET_CARD.md](DATASET_CARD.md) - Complete field documentation
- [datapackage.json](datapackage.json) - Frictionless Data descriptor
- [Wikipedia Source](https://en.wikipedia.org/wiki/List_of_serial_killers_by_number_of_victims)